# BYTE Manga Task — Inference

This notebook runs Qwen2.5-VL-3B-Instruct on the 15 test sequences and writes a JSONL submission. It uses the task dataset from Google Drive and saves outputs/checkpoints to `/content` to avoid Drive write-permission issues.

**Before running:** set up the Google Drive folder described below. Do not upload private task images or reference labels to a public repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, re
import torch
from PIL import Image

DATA = Path('/content/drive/MyDrive/Colab Notebooks/BYTE ML Task Data')
assert (DATA / 'sequences.json').exists(), f'Dataset not found at {DATA}'
assert (DATA / 'sample_submission.jsonl').exists(), 'sample_submission.jsonl not found'
print('Dataset folder found:', DATA)
print('CUDA available:', torch.cuda.is_available())

## 1. Install dependencies
Run this cell once per fresh Colab runtime.

In [ ]:
%pip -q install -U transformers accelerate bitsandbytes qwen-vl-utils pillow

## 2. Load the open-weight model

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print('Model loaded:', MODEL_ID)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory allocated:', round(torch.cuda.memory_allocated(0) / 1024**3, 2), 'GB')

## 3. Transcribe one page
This baseline uses `UNKNOWN` for all speakers; speaker identification is a known limitation.

In [ ]:
def transcribe_page(image_path):
    image = Image.open(DATA / image_path).convert('RGB')
    prompt_text = '''
Transcribe dialogue and meaningful narration from this single manga page.
Return ONLY valid JSON in this format:
{"entries": [{"speaker": "UNKNOWN", "text": "DIALOGUE"}]}
Rules:
- Copy only text clearly visible in this image; do not invent, paraphrase, or complete dialogue.
- Include readable speech bubbles and meaningful narration.
- Exclude sound effects, decorative titles, and credits.
- Keep each speech bubble together in one entry and preserve manga reading order.
- Preserve wording and punctuation; use uppercase text.
- Use UNKNOWN for every speaker.
- If there is no dialogue or narration, return {"entries": []}.
- Return valid JSON only, without markdown or explanation.
'''
    messages = [{ 'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': prompt_text},
    ]}]
    inputs = None
    try:
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[prompt], images=[image], return_tensors='pt').to('cuda')
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=900, do_sample=False)
        new_tokens = output[:, inputs['input_ids'].shape[1]:]
        response = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if not match:
            raise ValueError(f'Model did not return JSON for {image_path}: {response}')
        result = json.loads(match.group())
        if not isinstance(result.get('entries'), list):
            raise ValueError(f'JSON has no entries list for {image_path}')
        entries = []
        for item in result['entries']:
            if isinstance(item, dict) and item.get('text'):
                text = str(item['text']).strip().upper()
                if text:
                    entries.append({'speaker': 'UNKNOWN', 'text': text})
        return entries
    finally:
        image.close()
        if inputs is not None:
            del inputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 4. Generate the 15 test sequences
The checkpoint and final output are written to Colab's temporary storage (`/content`), not the read-only Drive folder. Download the final file before disconnecting the runtime.

In [ ]:
OUTPUT = Path('/content/test_predictions.jsonl')
CHECKPOINT = Path('/content/test_predictions_checkpoint.jsonl')

all_sequences = json.loads((DATA / 'sequences.json').read_text(encoding='utf-8'))
sample_rows = [json.loads(line) for line in (DATA / 'sample_submission.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
sequence_by_id = {s['sequence_id']: s for s in all_sequences}
test_sequences = [sequence_by_id[row['sequence_id']] for row in sample_rows]
assert len(test_sequences) == 15
assert all(len(s['images']) == 3 for s in test_sequences)

completed = {}
if CHECKPOINT.exists():
    for line in CHECKPOINT.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line)
            completed[row['sequence_id']] = row

for i, seq in enumerate(test_sequences, start=1):
    sid = seq['sequence_id']
    if sid in completed:
        print(f'[{i}/15] Already completed: {sid}')
        continue
    print(f'[{i}/15] Processing: {sid}')
    pages = []
    for page_num, image_path in enumerate(seq['images'], start=1):
        print(f'  Page {page_num}/3')
        pages.append(transcribe_page(image_path))
    completed[sid] = {'sequence_id': sid, 'pages': pages}
    with CHECKPOINT.open('w', encoding='utf-8') as f:
        for s in test_sequences:
            if s['sequence_id'] in completed:
                f.write(json.dumps(completed[s['sequence_id']], ensure_ascii=False) + '\n')

with OUTPUT.open('w', encoding='utf-8') as f:
    for sample in sample_rows:
        f.write(json.dumps(completed[sample['sequence_id']], ensure_ascii=False) + '\n')
print('Finished. Saved:', OUTPUT)
print('Sequences:', len(completed))

## 5. Validate and download the submission

In [ ]:
rows = [json.loads(line) for line in OUTPUT.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(rows) == 15
assert len({row['sequence_id'] for row in rows}) == 15
assert all(len(row['pages']) == 3 for row in rows)
assert all(isinstance(page, list) for row in rows for page in row['pages'])
print('Submission structure verified: 15 rows, 3 pages each.')
print('Example:', json.dumps(rows[0], ensure_ascii=False)[:500])

from google.colab import files
files.download(str(OUTPUT))